# TIDMAD: change one band, save it, and run your saved experiment

A model takes a noisy detector waveform and predicts its clean injected waveform. We use **band `0-3`**, so you can learn the configuration without setting up four separate band searches.

Follow this order: **locate your project → choose existing data or download → change settings → save → open the saved files → run their script in a terminal**. The notebook explains and prepares; it never calls an LLM or launches training. Keep both source repositories unchanged.

This notebook must be the copy in your external project. If you have not created one, run the following from the installed exp checkout first, then open the copied notebook with that checkout's Jupyter kernel:

```bash
export TUTORIAL_HOME=/absolute/path/to/my-tidmad-study
.venv/bin/python -B -m tutorials.paper.tidmad.project --project "$TUTORIAL_HOME" --infra-checkout /absolute/path/to/SIDERIUS-tutorial
.venv/bin/jupyter lab --ServerApp.root_dir="$TUTORIAL_HOME"
```

`TUTORIAL_HOME` is your study folder, not the exp repo and not a single run's output folder.

In [ ]:
import json
import os
import shlex
import sys
from pathlib import Path

PROJECT = globals().get(
    "PROJECT", Path(os.environ.get("TUTORIAL_HOME", Path.cwd())).resolve()
)
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
binding = json.loads((PROJECT / "project.json").read_text())
EXP, INFRA = Path(binding["exp_checkout"]), Path(binding["infra_checkout"])
assert Path(sys.prefix).resolve() == (EXP / ".venv").resolve(), (
    "Select the exp checkout's kernel"
)
os.chdir(EXP)
sys.dont_write_bytecode = True
from tasks.tidmad.runtime.file_split import FileSplit, sample_population
from tutorials.paper.tidmad.project import write_file_split_task, write_launcher
from tutorials.paper.tidmad.runner import TidmadExperiment, credential_status

base_path = PROJECT / "experiments/tidmad-experiment.json"
base = TidmadExperiment.model_validate_json(base_path.read_text())
print("YOUR PROJECT:", PROJECT)
print("Initial saved experiment:", base_path)
print("Initial task entry:", base.composition)
print("Initial launcher:", PROJECT / "scripts/run-tidmad.sh")

## 1. What are a task and an experiment on disk?

A **task package is a directory of related files**, not a single notebook. Its composition YAML is the entry file: it connects the data adapter, waveform contract, metric and Health rules. An **experiment is one JSON file**: it points to that task entry and sets budgets, model routing, data and output paths. A **launcher is one shell script**: its `EXPERIMENT` variable names the JSON it will read.

```text
YOUR_PROJECT/
├── notebooks/02_tidmad_tutorial.ipynb          your editable notebook
├── tasks/tidmad/                              the complete task package
│   ├── compositions/continuous_regression_frozen_pool.yaml  original paper pool
│   ├── compositions/file_holdout.yaml         created by the save step below
│   ├── declared/task_config_regression.yaml   waveform input/output contract
│   ├── resolved/dataset_profile.json          file names, channels, segment sizes
│   ├── resolved/metric_spec.json              what score means
│   ├── runtime/file_split.py                  implements your file membership
│   └── reference_data/                        fixed scoring references
├── experiments/tidmad-experiment.json         initial paper-pool demo
├── experiments/tidmad-file-split.json         created by the save step below
├── llm/agents.json                            model/provider choices, NO keys
├── advice/README.txt                          explains why advice is inactive
├── scripts/run-tidmad-file-split.sh           created by the save step below
├── scripts/test-tidmad-file-split.sh          separate final-test entrypoint
├── data/band-0-3/                             run inputs, NOT result files
├── runs/tidmad_file_split_001/                created only by a real launch
└── final-test/                               selected model and test results
```

The first save step creates the `file_holdout.yaml`, experiment and scripts together. Before that, those paths in this diagram are **planned filenames**, not files that already exist. Editing a notebook variable changes only kernel memory until you explicitly save it.

## 2. Choose ONE data entrance

**A — On the shared RTX 5090 machine:** use the existing `/home/klz/Data/TIDMAD`. The command below verifies the eight source files and creates eight small symbolic links in your project's `data/band-0-3/`. It does not download, copy, rename or alter the large HDF5 files. The local `segment_anchors.json` was supplied by project initialization. The links remain dependent on the shared source paths staying available.

**B — On your own machine:** obtain the official [TIDMAD downloader](https://github.com/jessicafry/TIDMAD) outside the source checkouts and download only four training and four validation files into your project's `data/band-0-3/`. Do not download the science dataset for this tutorial. One band still represents about 32 GB of uncompressed channel samples.

The next cell **prints** the command for your selected entrance; run that command once in a terminal. If your data is already prepared, skip preparation and inspect the paths instead. Both entrances produce the same `data_dir`, so the experiment and launch script do not need different implementations.

In [ ]:
DATA_ENTRY = "existing"  # "existing" on the shared machine; "download" elsewhere
if DATA_ENTRY == "existing":
    command = [
        str(EXP / ".venv/bin/python"),
        "-B",
        "-m",
        "tutorials.paper.data_entry",
        "--task",
        "tidmad",
        "--project",
        str(PROJECT),
        "--source",
        "/home/klz/Data/TIDMAD",
    ]
elif DATA_ENTRY == "download":
    command = [
        sys.executable,
        "/absolute/path/to/TIDMAD/download_data.py",
        "--output_dir",
        str(base.data_dir),
        "--train_files",
        "4",
        "--validation_files",
        "4",
        "--science_files",
        "0",
    ]
else:
    raise ValueError("Choose existing or download")
print("Run once from the exp checkout:", shlex.join(command))
print("Experiment data_dir:", base.data_dir)
for path in sorted(base.data_dir.glob("*")):
    print(path.name, "→", path.resolve() if path.is_symlink() else "local file")

## 3. Change the experiment: iterations, fractions and budgets

An **iteration** is another search cycle. An **epoch** is a pass through an attempt's selected training data. **Trial** tries a smaller workload and settings; **Formal** is the larger training/evaluation stage. Failures and retries mean these are not exact counts of successful runs.

| Change | JSON field | Meaning |
|---|---|---|
| Search longer | `iterations` | Search iteration ceiling |
| Train longer per attempt | `epochs` | Trial/Formal epoch ceiling |
| Trial training amount | `trial_train_fraction` | Portion of eligible training segments |
| Trial evaluation amount | `trial_val_fraction` | Portion of eligible validation segments |
| Formal training amount | `formal_train_fraction` | Portion of eligible training segments; paper-pool exception below |
| Formal evaluation amount | `formal_val_fraction` | Portion of eligible validation segments |
| Time allowance | `trial_minutes`, `formal_minutes` | Native per-stage training budgets, not a total script deadline |
| Memory allowance | `trial_vram_gib`, `formal_vram_gib` | GiB; leave GPU headroom |

Use fractions such as `.25`, not `25`. Changing a fraction samples fewer segments **inside an existing split**; it does not move a file from train to validation.

The original **paper-pool** task fixes 20 training PSD segments out of 200 per file. Trial `.5` means **10/20**. Formal `.1` declares the original **20/200** parent pool and uses all 20; it does not take another 10%. This entrypoint refuses other Formal training fractions for paper-pool. Evaluation is independent, over each original validation file's 200 segments. To freely change Formal fractions, use the new task in section 4.

In [ ]:
def variant(name, **changes):
    values = base.model_dump(mode="json")
    values.update(changes, run_name=name, workspace=str(PROJECT / "runs" / name))
    return TidmadExperiment.model_validate(values)


longer = variant("tidmad_longer_001", iterations=3, epochs=10)
less_trial = variant(
    "tidmad_less_trial_001", trial_train_fraction=0.25, trial_val_fraction=0.1
)
budgets = variant(
    "tidmad_budget_001",
    trial_minutes=3,
    formal_minutes=10,
    trial_vram_gib=6,
    formal_vram_gib=8,
)
for name, settings in [
    ("longer", longer),
    ("less-trial", less_trial),
    ("budgets", budgets),
]:
    print(name, "will save to", PROJECT / "experiments" / f"tidmad-{name}.json")
    print("  output after launch:", settings.workspace)

## 4. Change the task: hold out different file indices

For this simpler example, file indices represent different frequency ranges. Change the three lists below. The example uses **training files 0 and 1**, **validation file 2**, and **final-test file 3**:

| Role | Physical data read | Feedback |
|---|---|---|
| Training | `abra_training_0000.h5`, `abra_training_0001.h5` | Used to fit model weights |
| Workflow validation | `abra_validation_0002.h5` | Visible to the agent; can affect training/model selection |
| Final test | `abra_validation_0003.h5` | Separate inference after you select and fix one model |

Other family/index combinations are excluded from these scopes. This is **file-index / frequency-range holdout**, not a per-frequency label classification. It requires no FFT catalog or signal-strength threshold. Every original segment within each assigned file remains eligible, including weak-signal segments. File boundaries do not prove that exact frequencies or spectral content can never overlap.

Each list must be nonempty, the lists must not overlap, and together they must contain `0,1,2,3`. For example, changing validation to `[1]` also requires removing `1` from training and assigning `2` elsewhere. Do not rename or reorder raw files; scoring uses original file/segment identities.

The saved membership belongs to `tasks/tidmad/compositions/file_holdout.yaml`, under `task_data_path.config.split`. This new task uses ordinary scopes instead of the frozen paper pool. It changes the scientific protocol, so its scores are not a direct reproduction of the frozen paper experiment.

In [ ]:
TRAIN_FILES = [0, 1]
VALIDATION_FILES = [2]
FINAL_TEST_FILES = [3]
file_split = FileSplit(
    train_files=tuple(TRAIN_FILES),
    validation_files=tuple(VALIDATION_FILES),
    test_files=tuple(FINAL_TEST_FILES),
    seed=42,
)
print("The split that will be saved into your task YAML:")
print(file_split.model_dump_json(indent=2))
for role in ("train", "validation", "test"):
    print(
        role, {index: len(rows) for index, rows in file_split.population(role).items()}
    )

### Combine the new split with Trial/Formal proportions

Each assigned file contains 200 eligible PSD segments. With training files `[0,1]`, Trial `.25` selects 50 from each file (**100 total**); Formal `.5` selects 100 from each (**200 total**). With validation file `[2]`, Trial `.5` selects **100** and Formal `1.0` selects **200**. The final test remains all 200 segments of file 3 and is never part of these workflow scopes.

The custom adapter uses `ceil(fraction × eligible segments)` per file, at least one. A one-second PSD segment contains 10M samples, or 250 model windows of 40,000 samples. Smaller fractions can still fail the existing runtime calibration checks. Raise an inadequate allowance instead of disabling checks.

In [ ]:
for label, role, portion in [
    ("Trial training", "train", 0.25),
    ("Formal training", "train", 0.5),
    ("Trial validation", "validation", 0.5),
    ("Formal validation", "validation", 1.0),
]:
    counts = {
        i: len(rows)
        for i, rows in sample_population(
            file_split.population(role), portion, 42
        ).items()
    }
    print(label, counts, "total:", sum(counts.values()))

## 5. Save: this is where notebook edits become files

Set `SAVE_EXAMPLES=True` in the next cell and run it **once**. It creates the new task entry plus four experiment/script pairs. No training starts. Files are not silently overwritten; for a second study choose new filenames/run names (or edit the saved JSON directly in Jupyter).

After saving, changing `TRAIN_FILES` in an earlier cell alone does not update the YAML. Likewise, changing `budgets` in memory does not update a JSON already on disk. The launcher always reads the **saved** files. Save a new variant, or open and deliberately edit the saved file you intend to run.

The next section reads those files back so you can verify exactly what was saved.

In [ ]:
SAVE_EXAMPLES = False
saved = {}
if SAVE_EXAMPLES:
    task_entry = write_file_split_task(PROJECT / "tasks/tidmad", file_split)
    file_experiment = variant(
        "tidmad_file_split_001",
        protocol="file-holdout",
        composition=str(task_entry),
        trial_train_fraction=0.25,
        trial_val_fraction=0.5,
        formal_train_fraction=0.5,
        formal_val_fraction=1.0,
    )
    for name, settings in [
        ("longer", longer),
        ("less-trial", less_trial),
        ("budgets", budgets),
        ("file-split", file_experiment),
    ]:
        config = PROJECT / "experiments" / f"tidmad-{name}.json"
        script = PROJECT / "scripts" / f"run-tidmad-{name}.sh"
        with config.open("x") as f:
            f.write(settings.model_dump_json(indent=2))
        write_launcher(script, config, INFRA)
        saved[name] = config
    write_launcher(
        PROJECT / "scripts/test-tidmad-file-split.sh",
        saved["file-split"],
        INFRA,
        final=True,
    )
    print("Saved task:", task_entry)
for name in ("longer", "less-trial", "budgets", "file-split"):
    path = PROJECT / "experiments" / f"tidmad-{name}.json"
    print("SAVED" if path.exists() else "NOT SAVED", path)

## 6. Open the saved files and follow their connections

In JupyterLab's left file browser, open `experiments/tidmad-file-split.json`. Its `composition` points to `tasks/tidmad/compositions/file_holdout.yaml`. Open that YAML to see the three file lists. Then open `scripts/run-tidmad-file-split.sh`; its `EXPERIMENT` line points back to your JSON. These three files form the runnable configuration.

After saving, you can also open [the experiment JSON](../experiments/tidmad-file-split.json), [the task entry YAML](../tasks/tidmad/compositions/file_holdout.yaml), and [the search script](../scripts/run-tidmad-file-split.sh) from this notebook. Links do not work before those files are created.

The following cell prints their actual contents from disk. It does not display credentials: routing contains model/provider names only, and keys stay in the environment.

In [ ]:
SELECTED_DEMO = "file-split"  # or "longer", "less-trial", "budgets"
config_path = PROJECT / "experiments" / f"tidmad-{SELECTED_DEMO}.json"
script_path = PROJECT / "scripts" / f"run-tidmad-{SELECTED_DEMO}.sh"
selected = None
if config_path.is_file() and script_path.is_file():
    selected = TidmadExperiment.model_validate_json(config_path.read_text())
    print("EXPERIMENT FILE:", config_path, "\n", config_path.read_text())
    print("TASK ENTRY:", selected.composition, "\n", selected.composition.read_text())
    print("LAUNCH SCRIPT:", script_path, "\n", script_path.read_text())
else:
    print("Not saved yet. Run section 5 with SAVE_EXAMPLES=True, then return here.")

## 7. Run the script that points to the experiment you just inspected

The initial `scripts/run-tidmad.sh` still runs the **initial paper-pool** experiment. It does not automatically switch to your new file split. For the new split, run `scripts/run-tidmad-file-split.sh` instead. The next cell prints the exact absolute command and destination from the saved experiment.

First preview without `--launch`; then use `--launch` in a terminal when ready for paid API calls and GPU work. The notebook only prints commands.

Before launch, export the providers' required keys in that terminal. Use a trusted external secret store or a mode-600 environment file outside both repositories and this study; do not paste keys in notebook cells, scripts, JSON or logs. A credential file existing on disk is not enough: the terminal must source/export it. Notebook and terminal environments can differ. Presence checks show names/booleans, not values or proof of working provider access.

`llm/agents.json` chooses providers/models. Human advice and data analysis are disabled by this NoPrior entrypoint; literature review remains enabled. `advice/README.txt` explains the boundary—placing an advice file there does not activate it.

In [ ]:
if selected is not None:
    print("1. Preview:", shlex.join(["bash", str(script_path)]))
    print("2. Launch:", shlex.join(["bash", str(script_path), "--launch"]))
    print("Reads experiment:", config_path)
    print("Reads task:", selected.composition)
    print("Reads data:", selected.data_dir)
    print("Writes run output:", selected.workspace)
    print(
        "Writes launch receipt:",
        selected.workspace.with_name(selected.workspace.name + ".tutorial.json"),
    )
for name, present in credential_status(selected or base).items():
    print(
        name,
        "present"
        if present
        else "MISSING — export in launching terminal; launch will refuse",
    )

The launcher accepts one RTX 5090 or H100 with working CUDA. Change `gpu` and budgets in the saved experiment, leaving memory headroom. Another NVIDIA GPU requires extending/qualifying the GPU check and validating the pinned CUDA environment with a small run; changing the name alone is not qualification. AMD and Intel GPUs are unsupported.

This fixed workflow runs under your local account without a separate security sandbox. Its time budgets are not total wall-time or API-cost caps. Existing blocking Health rules stay active; results are diagnostic. A fresh launch requires a new `workspace` and `run_name`, so rerunning a script against an existing output directory is refused.

## 8. Final test: use file 3 only after selecting one completed model

Workflow validation can guide the agent and epoch/model selection. Final test must not. Stop search, select one completed model using validation plus Health evidence, and save `final-test/candidate.json` in your project:

```json
{
  "model_name": "YOUR_SELECTED_MODEL_REGISTRY_NAME",
  "exp_id": "YOUR_SELECTED_ATTEMPT_ID",
  "checkpoint": "/YOUR_PROJECT/runs/tidmad_file_split_001/PATH_TO_SELECTED.pth",
  "model_config_path": "/YOUR_PROJECT/runs/tidmad_file_split_001/PATH_TO_MODEL_CONFIG.json",
  "loss_config_path": "/YOUR_PROJECT/runs/tidmad_file_split_001/PATH_TO_LOSS_CONFIG.json",
  "generated_library": "/YOUR_PROJECT/runs/tidmad_file_split_001/generated_library",
  "search_completed": true,
  "inference_batch_size": 1
}
```

Replace placeholders with the selected attempt's actual artifacts/native inference command; do not choose checkpoints by modification time. Keep `_OK_<exp_id>`, the native training-success marker, next to the checkpoint. In a terminal:

```bash
/YOUR_PROJECT/scripts/test-tidmad-file-split.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --seal
/YOUR_PROJECT/scripts/test-tidmad-file-split.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --evaluate
```

`--seal` records selection hashes without test inference. `--evaluate` requires the same model/config/task, runs inference without LLM calls or training, then uses the selected task's metric and scoreability checks. Open `final-test/test-001/result.json` to see the separate test result. It does **not** run the full workflow Health assessment and is not a paper-qualified score.

A started test cannot silently be retried in that output directory. The holdout is procedural, not an OS access barrier: files remain locally readable. Do not feed test results back into model selection; if you adapt to them, you need a new final holdout.

## 9. After launch, where do I look?

Open the `workspace` printed in section 7. It is different from your project root. The adjacent `.tutorial.json` records the exact configuration and command; the run folder contains node records, generated models and calibration evidence. Terminal output stays in that terminal unless you redirect it to your project-local log.

Check status, Trial/Formal role, metric and Health evidence together. A completed command does not imply a useful model or a scientifically accepted score. The final test has its own `final-test/test-001/result.json`; it is not added to the search history.

In [ ]:
if selected is not None:
    records = (
        sorted(selected.workspace.rglob("run_output_*.json"))
        if selected.workspace.exists()
        else []
    )
    print("Search result records:", *records, sep="\n")
    if not records:
        print("No completed records at", selected.workspace, "yet.")
print(
    "Final-test reports:",
    *sorted((PROJECT / "final-test").glob("*/result.json")),
    sep="\n",
)